# Apex Predictor — Deep Learning: TabNet

Terzo esperimento. TabNet è un'architettura pensata apposta per dati tabellari: usa un meccanismo di attenzione sequenziale per imitare concettualmente il comportamento di un albero, restando però una rete neurale allenabile end-to-end. 
Confronto di riferimento: XGBoost F1 0.717, MLP F1 0.657, LSTM F1 0.678.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

from src.data_loading import load_raw_data, build_working_dataset
from src.features import build_all_features
from src.train import FEATURE_COL, temporal_split
from src.dl_common import find_best_threshold_np

torch.manual_seed(42)
np.random.seed(42)

data = load_raw_data()
df = build_working_dataset(data["races"], data["results"], min_year=2004)
df = build_all_features(
    df, data["circuits"], data["drivers"], data["constructors"],
    data["driver_standings"], data["constructor_standings"], data["qualifying"]
)

train_df, test_df = temporal_split(df)

# TabNet lavora meglio con feature normalizzate, a differenza di XGBoost che è invariante alla scala
feat_mean = train_df[FEATURE_COL].mean()
feat_std = train_df[FEATURE_COL].std().replace(0, 1)

X_train = ((train_df[FEATURE_COL] - feat_mean) / feat_std).values.astype(np.float32)
X_test = ((test_df[FEATURE_COL] - feat_mean) / feat_std).values.astype(np.float32)
y_train = train_df["podium"].astype(int).values
y_test = test_df["podium"].astype(int).values

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (8519, 14), Test: (759, 14)


In [2]:
# TabNetClassifier ha un'interfaccia simile a scikit-learn (fit/predict)
model = TabNetClassifier(
    n_d=16,              # dimensione della rappresentazione "decisionale" ad ogni step, 16 è un valore contenuto, adatto a un dataset di questa dimensione 
    n_a=16,              # dimensione della rappresentazione usata per decidere quali feature guardare allo step successivo
    n_steps=3,           # quanti "step" di attenzione sequenziale
    gamma=1.5,           # controlla quanto una feature già usata in uno step può essere riutilizzata negli step successivi
    lambda_sparse=1e-3,  # penalizza l'uso di troppe feature contemporaneamente, spingendo il modello verso maschere di attenzione più sparse (quindi più interpretabili)
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params=dict(step_size=10, gamma=0.9),  # riduce il learning rate nel tempo
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=42,
    verbose=1,
)

model.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_test, y_test)],
    eval_metric=["logloss"],
    max_epochs=100,
    patience=15,       # stesso principio di early stopping già usato per MLP/LSTM
    batch_size=256,
    weights=1,          # 1 = bilanciamento automatico delle classi
)

/home/daniele/progetti/apex-predictor/venv/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.47601 | val_0_logloss: 1.27858 |  0:00:04s
epoch 1  | loss: 0.35426 | val_0_logloss: 0.90872 |  0:00:12s
epoch 2  | loss: 0.34047 | val_0_logloss: 0.52084 |  0:00:19s
epoch 3  | loss: 0.33218 | val_0_logloss: 0.42368 |  0:00:25s
epoch 4  | loss: 0.34794 | val_0_logloss: 0.38523 |  0:00:30s
epoch 5  | loss: 0.32527 | val_0_logloss: 0.35122 |  0:00:36s
epoch 6  | loss: 0.32947 | val_0_logloss: 0.49563 |  0:00:43s
epoch 7  | loss: 0.33262 | val_0_logloss: 0.38991 |  0:00:50s
epoch 8  | loss: 0.33207 | val_0_logloss: 0.49252 |  0:00:57s
epoch 9  | loss: 0.3187  | val_0_logloss: 0.31063 |  0:01:04s
epoch 10 | loss: 0.32893 | val_0_logloss: 0.37725 |  0:01:10s
epoch 11 | loss: 0.3213  | val_0_logloss: 0.39852 |  0:01:16s
epoch 12 | loss: 0.32989 | val_0_logloss: 0.27191 |  0:01:25s
epoch 13 | loss: 0.32126 | val_0_logloss: 0.31206 |  0:01:38s
epoch 14 | loss: 0.32104 | val_0_logloss: 0.34447 |  0:01:53s
epoch 15 | loss: 0.31628 | val_0_logloss: 0.41137 |  0:02:01s
epoch 16

/home/daniele/progetti/apex-predictor/venv/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [3]:
y_proba = model.predict_proba(X_test)[:, 1]

best = find_best_threshold_np(y_test, y_proba)
print(f"TabNet — soglia {best['threshold']:.2f}: "
      f"precision {best['precision']:.3f}, recall {best['recall']:.3f}, F1 {best['f1']:.3f}")
print(f"\nConfronto — XGBoost: F1 0.717 | MLP: F1 0.657 | LSTM: F1 0.678")

TabNet — soglia 0.60: precision 0.656, recall 0.775, F1 0.711

Confronto — XGBoost: F1 0.717 | MLP: F1 0.657 | LSTM: F1 0.678


## Interpretabilità: confronto con SHAP (XGBoost)

TabNet ha attenzione nativa, possiamo vedere quali feature ha "guardato" di più, senza bisogno di un metodo post-hoc come SHAP

In [4]:
tabnet_importance = pd.DataFrame({
    "feature": FEATURE_COL,
    "tabnet_importance": model.feature_importances_
}).sort_values("tabnet_importance", ascending=False)

print(tabnet_importance.to_string(index=False))

# Confronto diretto con l'ordine SHAP trovato su XGBoost
shap_ranking_xgboost = [
    "grid", "driver_recent_position_avg", "qualifying_gap_seconds",
    "driver_recent_points_avg", "driver_standing_position",
    "constructor_standing_position", "driver_circuit_avg_position",
    "circuit_overtaking_index", "teammate_position_gap",
    "race_precipitation_mm", "constructor_reliability", "race_max_temp_c",
    "circuit_avg_speed_history", "circuit_num_corners"
]

tabnet_ranking = tabnet_importance["feature"].tolist()

print("\nConfronto ranking (posizione 1 = più importante):")
comparison_ranking = pd.DataFrame({
    "feature": FEATURE_COL,
    "rank_XGBoost_SHAP": [shap_ranking_xgboost.index(f) + 1 for f in FEATURE_COL],
    "rank_TabNet": [tabnet_ranking.index(f) + 1 for f in FEATURE_COL],
}).sort_values("rank_XGBoost_SHAP")
print(comparison_ranking.to_string(index=False))

                      feature  tabnet_importance
     driver_recent_points_avg           0.288346
                         grid           0.243979
   driver_recent_position_avg           0.165847
     driver_standing_position           0.110809
       qualifying_gap_seconds           0.050955
      constructor_reliability           0.045577
constructor_standing_position           0.041047
  driver_circuit_avg_position           0.027392
              race_max_temp_c           0.012133
     circuit_overtaking_index           0.006836
          circuit_num_corners           0.004903
        race_precipitation_mm           0.001162
        teammate_position_gap           0.000754
    circuit_avg_speed_history           0.000260

Confronto ranking (posizione 1 = più importante):
                      feature  rank_XGBoost_SHAP  rank_TabNet
                         grid                  1            2
   driver_recent_position_avg                  2            3
       qualifying_gap_second